# FinTrust ML Engineering Track — Week 2 Intern Walkthrough
### Project: FinTrust ML Workflow — Initial Implementation
**Mentor Demonstration Notebook**

---

## 📚 Session Learning Objectives
By the end of this walkthrough, interns will understand:
1. **Production ML Directory Structure**: How to organize code modularly instead of using monolithic notebooks.
2. **Data Contracts & Validation (Part B)**: Catching nulls, type mismatches, out-of-bounds numbers, and unauthorized categories before inference.
3. **Reproducible Preprocessing (Part C)**: Preventing data leakage using Scikit-Learn  and handling unseen categorical levels.
4. **The 7-Stage Prediction Pipeline (Part D)**: Tracing: .
5. **Automated Quality Assurance (Part E)**: Writing deterministic unit/integration tests.
6. **Model Serving Concepts (Advanced)**: Packaging ML workflows into REST APIs with FastAPI.

## 1. Environment & Setup
We set up module paths and import our production modules from .

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import src.*
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from src.config import PATHS, SCHEMA, MODEL_CONFIG
from src.validation.validator import DataValidator, DataValidationError
from src.preprocessing.pipeline import FinTrustDataPreprocessor, TemporalFeatureExtractor
from src.models.predict import FinTrustPredictionPipeline

print("✓ Environment initialized successfully!")
print(f"Raw Data Path: {PATHS.raw_transaction_path}")

✓ Environment initialized successfully!
Raw Data Path: /Users/prestige/Workspace/ANALYSTLAB ML WEEK 2 LAB/data/raw/FinTrust_Transaction_Data.xlsx


## 2. Data Exploration & Schema Understanding
Let us inspect the raw transaction dataset from FinTrust.

In [8]:
# Load raw transaction data
df_raw = pd.read_excel(PATHS.raw_transaction_path)
display(f"Total transactions: {len(df_raw):,} | Total columns: {df_raw.shape[1]}")
display(df_raw.head(5))

# Target variable distribution
risk_counts = df_raw["Risk_Review_Flag"].value_counts()
print("Target Variable (Risk_Review_Flag) Breakdown:")
for label, count in risk_counts.items():
    print(f"  {label}: {count:,} ({count/len(df_raw)*100:.2f}%)")

'Total transactions: 12,000 | Total columns: 11'

,Transaction_ID,Customer_ID,Transaction_DateTime,Transaction_Type,Amount_NGN,Channel,Device_Type,Location,International_Transaction,Transaction_Status,Risk_Review_Flag
0,FT-T000001,FT-C01135,2026-01-01 00:00:00,Card Purchase,5923.80,Mobile App,Android,Enugu,No,Reversed,Yes
1,FT-T000002,FT-C00428,2026-01-01 00:10:00,Cash Withdrawal,11681.12,ATM,Android,Kaduna,No,Successful,Yes
2,FT-T000003,FT-C00469,2026-01-01 00:21:00,Transfer,571.91,ATM,POS Terminal,Kano,No,Successful,No
3,FT-T000004,FT-C01015,2026-01-01 00:32:00,Deposit,105367.52,Mobile App,Web Browser,Abuja,No,Successful,No
4,FT-T000005,FT-C00870,2026-01-01 00:43:00,Cash Withdrawal,3049.06,Mobile App,POS Terminal,Abuja,No,Successful,No


Target Variable (Risk_Review_Flag) Breakdown:
  No: 9,648 (80.40%)
  Yes: 2,352 (19.60%)


## 3. Part B: Data Validation in Action
Let us test how handles both clean and intentionally malformed data.

In [ ]:
# 1. Validation on real training data (Lenient mode)
validator = DataValidator(schema=SCHEMA, strict=False)
is_valid, report, df_clean = validator.validate(df_raw, is_training=True)
report.print_report()

# 2. Testing failure cases: what happens with corrupted data?
corrupted_record = pd.DataFrame([{
    "Transaction_ID": "FT-T999",
    "Customer_ID": "FT-C999",
    "Transaction_DateTime": 46023.0,
    "Transaction_Type": "Transfer",
    "Amount_NGN": -5000.0,       # ILLEGAL: Negative amount!
    "Channel": "Telepathy",      # ILLEGAL: Unrecognized category!
    "Device_Type": "Android",
    "Location": "Lagos",
    "International_Transaction": "No",
    "Transaction_Status": "Successful"
}])

strict_validator = DataValidator(schema=SCHEMA, strict=True)
try:
    strict_validator.validate(corrupted_record)
except DataValidationError as e:
    print("STRICT VALIDATION CAUGHT ANOMALIES:")
    for err in e.report.errors:
        print(f"   - [{err.error_type}] {err.field}: {err.message}")


==================== DATA VALIDATION REPORT: PASSED ====================
Total Records Inspected: 12000
Errors: 0 | Warnings: 2
Pipeline Action: PROCEED

Non-Critical Warnings (Monitored/Imputed):
  [!] Field: Device_Type | Type: MISSING_VALUES
      Message: Column 'Device_Type' has 96 (0.80%) missing values. Downstream imputation will be applied.
  [!] Field: Location | Type: MISSING_VALUES
      Message: Column 'Location' has 96 (0.80%) missing values. Downstream imputation will be applied.

🛑 STRICT VALIDATION CAUGHT ANOMALIES:
   - [UNEXPECTED_CATEGORY] Channel: Column 'Channel' contains unexpected categories: ['Telepathy']
   - [OUT_OF_RANGE] Amount_NGN: Found 1 transactions with invalid Amount_NGN (<= 0 or > 50000000.0).


## 4. Part C: Preprocessing & Leakage Prevention
Here we show how  extracts cyclical time signals and how  protects against unseen production categories.

In [10]:
# Step 1: Temporal signal extraction
extractor = TemporalFeatureExtractor()
df_temporal = extractor.transform(df_raw.head(5))
print("Engineered temporal features:")
display(df_temporal[["Transaction_DateTime", "Transaction_Hour", "Transaction_DayOfWeek", "Is_Weekend"]])

# Step 2: Testing unseen categories handling
preprocessor = FinTrustDataPreprocessor.load(PATHS.preprocessor_artifact_path)
test_sample = df_raw.head(1).copy()
test_sample["Channel"] = "SmartFridge"  # Category never seen in training!

matrix_out = preprocessor.transform(test_sample)
print(f" Successfully transformed unseen category without error! Output shape: {matrix_out.shape}")

Engineered temporal features:


,Transaction_DateTime,Transaction_Hour,Transaction_DayOfWeek,Is_Weekend
0,2026-01-01 00:00:00,0,3,0
1,2026-01-01 00:10:00,0,3,0
2,2026-01-01 00:21:00,0,3,0
3,2026-01-01 00:32:00,0,3,0
4,2026-01-01 00:43:00,0,3,0


 Successfully transformed unseen category without error! Output shape: (1, 36)


## 5. Part D: The 7-Stage Prediction Pipeline
Let us trace a single transaction through all 7 stages:
**Data -> Validation -> Preprocessing -> Feature Prep -> Model -> Prediction -> Output**

In [11]:
pipeline = FinTrustPredictionPipeline(strict_validation=True)

# High-risk nocturnal international transfer
suspicious_transaction = {
    "Transaction_ID": "FT-T999123",
    "Customer_ID": "FT-C00452",
    "Transaction_DateTime": 46023.125,  # 03:00 AM
    "Transaction_Type": "Transfer",
    "Amount_NGN": 850000.0,            # High value transfer (NGN 850,000)
    "Channel": "Web",
    "Device_Type": "Web Browser",
    "Location": "Lagos",
    "International_Transaction": "Yes", # International
    "Transaction_Status": "Successful"
}

# Run pipeline with full verbose logging
scored_output = pipeline.run_pipeline(suspicious_transaction, verbose=True)

print("FINAL ENRICHED BANKING DECISION:")
display(scored_output[["Transaction_ID", "Predicted_Risk_Flag", "Risk_Probability", "Risk_Tier", "Operational_Action"]])


🚀 EXECUTING FINTRUST 7-STAGE PREDICTION WORKFLOW
[STAGE 1: DATA INGESTION]
  -> Ingested 1 transaction record(s).
  -> Ingested columns: ['Transaction_ID', 'Customer_ID', 'Transaction_DateTime', 'Transaction_Type', 'Amount_NGN', 'Channel', 'Device_Type', 'Location', 'International_Transaction', 'Transaction_Status']

[STAGE 2: DATA VALIDATION]
  -> Validation Passed: True
  -> Action Taken: PROCEED
  -> Errors Detected: 0 | Warnings: 0

[STAGE 3: PREPROCESSING]
  -> Applying temporal feature extraction (Hour, DayOfWeek, Is_Weekend)...
  -> Imputing missing values with training medians/modes...

[STAGE 4: FEATURE PREPARATION]
  -> Generated model-ready feature matrix.
  -> Feature matrix shape: (1, 36) (Rows x One-Hot Features)

[STAGE 5: MODEL INFERENCE]
  -> Evaluated Random Forest Classifier.
  -> Generated raw risk probabilities in range [0.744, 0.744].

[STAGE 6: PREDICTION & DECISION LOGIC]
  -> Applying operational decision threshold: 0.35

[STAGE 7: OUTPUT GENERATION]
  -> Asse

,Transaction_ID,Predicted_Risk_Flag,Risk_Probability,Risk_Tier,Operational_Action
0,FT-T999123,Yes,0.7441,High,Immediate Review / Payment Hold


## 6. Part E: Technical Tests Execution
We can execute the automated test suite directly to verify all 12 tests pass.

In [12]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "../tests/", "-v"], capture_output=True, text=True)
print(result.stdout)

============================= test session starts ==============================
platform darwin -- Python 3.14.6, pytest-9.1.1, pluggy-1.6.0 -- /Users/prestige/Workspace/ANALYSTLAB ML WEEK 2 LAB/venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/prestige/Workspace/ANALYSTLAB ML WEEK 2 LAB
plugins: anyio-4.15.1
collecting ... collected 12 items

../tests/test_prediction.py::test_pipeline_valid_single_prediction PASSED [  8%]
../tests/test_prediction.py::test_pipeline_rejects_corrupted_data PASSED [ 16%]
../tests/test_prediction.py::test_pipeline_reproducibility PASSED        [ 25%]
../tests/test_preprocessing.py::test_temporal_feature_extraction PASSED  [ 33%]
../tests/test_preprocessing.py::test_unseen_categories_handling PASSED   [ 41%]
../tests/test_preprocessing.py::test_missing_value_imputation PASSED     [ 50%]
../tests/test_validation.py::test_valid_input PASSED                     [ 58%]
../tests/test_validation.py::test_missing_values_warning_and_error PASSED [ 66%]
../te

## 7. Summary & Mentor Takeaways for Interns
- **Modularity:** Keep validation, transformation, modeling, and serving isolated.
- **Defensive Engineering:** Always validate at system borders.
- **Fair Evaluation:** Use stratified splits and recall-focused metrics for imbalanced datasets.
- **Reproducibility:** Version everything: code (git), dependencies (), and artifacts ().